### Görev1 Veri Hazırlama

In [23]:
import urllib.request
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
urllib.request.urlretrieve(url, "input.txt")

('input.txt', <http.client.HTTPMessage at 0x26912a70b00>)

In [24]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [25]:
print("karakter uzunluğu:", len(text))

karakter uzunluğu: 1115394


In [26]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [27]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65


In [28]:
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

print(encode("hii there"))


[46, 47, 47, 1, 58, 46, 43, 56, 43]


In [29]:
print(decode([15, 46, 47, 50, 50, 43, 56]))

Chiller


In [30]:
import torch
import torch.nn as nn
from torch.nn import functional as F

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [31]:
n =  int(0.9*len(data))
train_data = data[:n] # %90 eğitim %10 doğrulama
val_data = data[n:]

In [32]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [33]:
x = train_data[:block_size]
y = train_data[1:block_size+1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"input:{context} target:{target}")

input:tensor([18]) target:47
input:tensor([18, 47]) target:56
input:tensor([18, 47, 56]) target:57
input:tensor([18, 47, 56, 57]) target:58
input:tensor([18, 47, 56, 57, 58]) target:1
input:tensor([18, 47, 56, 57, 58,  1]) target:15
input:tensor([18, 47, 56, 57, 58,  1, 15]) target:47
input:tensor([18, 47, 56, 57, 58,  1, 15, 47]) target:58


In [34]:
torch.manual_seed(1337)
batch_size = 4
block_size = 8

def get_batch(split):

    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+1+block_size] for i in ix])
    return x, y

xb, yb = get_batch('train')

print('inputs:')
print(xb.shape)
print(xb)

print('targets:')
print(yb.shape)
print(yb)

print('_____________')

for b in range(batch_size):

    for t in range(block_size):
        for t in range(block_size):
            context = xb[b, :t+1]
            target = yb[b,t]
            print(f"input{context.tolist()} target:{target}")

inputs:
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets:
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
_____________
input[24] target:43
input[24, 43] target:58
input[24, 43, 58] target:5
input[24, 43, 58, 5] target:57
input[24, 43, 58, 5, 57] target:1
input[24, 43, 58, 5, 57, 1] target:46
input[24, 43, 58, 5, 57, 1, 46] target:43
input[24, 43, 58, 5, 57, 1, 46, 43] target:39
input[24] target:43
input[24, 43] target:58
input[24, 43, 58] target:5
input[24, 43, 58, 5] target:57
input[24, 43, 58, 5, 57] target:1
input[24, 43, 58, 5, 57, 1] target:46
input[24, 43, 58, 5, 57, 1, 46] target:43
input[24, 43, 58, 5, 57, 1, 46, 43] target:39
input[24] target:43
input[24, 43] target:58
input[24, 43, 58] target:

### biagram model

In [35]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)




    def forward(self, idx, targets = None):
        logits = self.token_embedding_table(idx)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss


    
    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx, targets=None)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx



m = BigramLanguageModel(vocab_size)

logits, loss = m(xb, yb)
print(logits.shape)
print(loss)
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))

torch.Size([32, 65])
tensor(5.0364, grad_fn=<NllLossBackward0>)

lfJeukRuaRJKXAYtXzfJ:HEPiu--sDioi;ILCo3pHNTmDwJsfheKRxZCFs
lZJ XQc?:s:HEzEnXalEPklcPU cL'DpdLCafBheH


In [36]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3) #PyTorch Optimizer

In [37]:
batch_size = 32
for steps in range(100):
    xb, yb = get_batch('train')

    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    print(loss.item())

4.647705078125
4.722455024719238
4.7174859046936035
4.6955246925354
4.676054954528809
4.6826934814453125
4.67878532409668
4.65626859664917
4.60831880569458
4.61800479888916
4.576215744018555
4.73443603515625
4.60615348815918
4.540811538696289
4.590214252471924
4.597837924957275
4.573626518249512
4.599429130554199
4.612548351287842
4.6582255363464355
4.647839069366455
4.584323883056641
4.5858540534973145
4.577819347381592
4.654167652130127
4.679945468902588
4.568190574645996
4.636806488037109
4.593273162841797
4.6050705909729
4.542919635772705
4.583306312561035
4.681209564208984
4.553791046142578
4.628030776977539
4.502110004425049
4.617300987243652
4.559780120849609
4.582223892211914
4.55769157409668
4.660159111022949
4.530014514923096
4.597833633422852
4.525141716003418
4.486090183258057
4.622176647186279
4.4698638916015625
4.6170148849487305
4.606544017791748
4.591933727264404
4.646505355834961
4.568349838256836
4.622153282165527
4.527293682098389
4.59453010559082
4.686464309692383
4

In [38]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')

    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
print(loss.item())

2.502596855163574


In [39]:
print(decode(m.generate(idx, max_new_tokens=500)[0].tolist()))



I maise pu l t t m!
TMI ty nnt thathan ot, thased t RWhainofohe.
IN s EGHo t gdis hry at bepoirerrean; sin t, lo F:
s!
I ccthe CAl ow t s sou mun coknuf hond'sus s mand
Yowhrouarfoar eanorendapuge he bd y co heatereave that.
KIUCUMBUR:
II be thator fown pe thyor bld CARD t; sty.
HEOUMyove I mu't s ngowite weeare Itef pe atour f cor t Corofe VOushe d t me mesandepr-centhigoorethit b tulu

MPEED: m;
Ar lon rus me sonon p lk miz,
UENCy, sthe.

I f lofim, an m b thoke q'setchind th yes hache:
NRLI'


In [ ]:
@torch.no_grad()
def estimate_loss():
    out = {}
    m.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(200)
        for k in range(200):
            X, Y = get_batch(split)
            logits, loss = m(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    m.train()
    return out

print(estimate_loss())  

{'train': tensor(2.4683), 'val': tensor(2.5020)}


### Görev 2: Geçmişin Ortalamasını alma.

In [41]:
#Self Attention trick

torch.manual_seed(1337)
B,T,C = 4,8,2

x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

## for döngüsü

In [42]:
xbow = torch.zeros((B,T,C))  #bow = bag of words


for b in range(B):
    for t in range(T):
        xprev = x[b, :t+1]
        xbow[b,t] = torch.mean(xprev, 0)
x[0]      
  

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [43]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

# Lower Triangular Matris

In [44]:
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)

xbow2 = wei @ x # (B, T, T)  @ (B, T, C) ->>>  (B, T, C)
torch.allclose(xbow, xbow2, atol=1e-6)

True

`torch.allclose(xbow, xbow2)` sonucu `False` döndü. Aradaki farkı kontrol etmek için:

```python
print((xbow - xbow2).abs().max())
```

çıktısı:

```text
tensor(3.2363e-08)
```

oldu. Bu çok küçük fark, kayan noktalı sayıların hesaplanmasından kaynaklanabilir.

Bu nedenle toleransı artırarak:

```python
torch.allclose(xbow, xbow2, atol=1e-6)
```

kullandım ve sonuç `True` oldu. Yani iki tensör arasındaki fark, `1e-6` toleransının altında.


In [45]:
torch.manual_seed(42)

a = torch.tril(torch.ones(3,3))
a = a / torch.sum(a, 1, keepdim=True)
b = torch.randint(0,10,(3,2)).float()
c = a @ b

print("a=")
print(a)
print("b=")
print(b)

print('-----')
print("c=")
print(c)

a=
tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])
b=
tensor([[2., 7.],
        [6., 4.],
        [6., 5.]])
-----
c=
tensor([[2.0000, 7.0000],
        [4.0000, 5.5000],
        [4.6667, 5.3333]])


### softmax

In [46]:
tril = torch.tril(torch.ones(T, T))

wei = torch.zeros((T, T))

wei = wei.masked_fill(tril == 0, float('-inf'))

wei = F.softmax(wei, dim=-1)

xbow3 = wei @ x

torch.allclose(xbow, xbow3, atol=1e-6)

True

kase örneği

# Görev 3: Self-attention head

In [97]:
torch.manual_seed(1337)
B,T,C = 4,8,32  #batch, time, channel

x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x) # B, T, 16 (HEAD SIZE)
q = query(x) # B, T, 16 (HEAD SIZE)
wei = q @ k.transpose(-2, -1)  # (B, T, 16) @ (B, 16, T) ----->> (B, T, T)



tril = torch.tril(torch.ones(T, T))
#wei = torch.zeros((T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v

out.shape

wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In [98]:
print(wei[0, -1])  # son tokenın (8. pozisyon) geçmişe bakış ağırlıkları

tensor([0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391],
       grad_fn=<SelectBackward0>)


In [103]:
k = torch.randn(B, T, head_size)
q = torch.randn(B, T, head_size)
wei = q @ k.transpose(-2,-1) # * head_size **-05

wei_karekok = q @ k.transpose(-2,-1) * head_size ** -0.5



In [105]:
print("Bölünmemiş wei varyansı:", wei.var().item())
print("√head_size'a bölünmüş wei varyansı:", wei_karekok.var().item())


print("\nBölünmeden softmax (ilk satır):")
print(F.softmax(wei[0,0], dim=-1))

print("\n√head_size'a bölerek softmax (ilk satır):")
print(F.softmax(wei_karekok[0,0], dim=-1))

Bölünmemiş wei varyansı: 15.931692123413086
√head_size'a bölünmüş wei varyansı: 0.9957307577133179

Bölünmeden softmax (ilk satır):
tensor([7.7338e-01, 1.7740e-04, 1.5693e-02, 1.2087e-03, 2.4352e-03, 4.8160e-02,
        1.5851e-01, 4.3435e-04])

√head_size'a bölerek softmax (ilk satır):
tensor([0.3065, 0.0377, 0.1157, 0.0609, 0.0726, 0.1531, 0.2062, 0.0472])


# Görev 4: Self Attention Head'i Biagrama Ekleme

In [107]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        v = self.value(x)
        out = wei @ v
        return out


class BigramLanguageModelWithAttention(nn.Module):
    def __init__(self, vocab_size, n_embed):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        self.sa_head = Head(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T))
        x = tok_emb + pos_emb
        x = self.sa_head(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx


@torch.no_grad()
def estimate_loss(model):
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(200)
        for k in range(200):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out


n_embed = 32
m2 = BigramLanguageModelWithAttention(vocab_size, n_embed)
optimizer2 = torch.optim.AdamW(m2.parameters(), lr=1e-3)

batch_size = 32
for steps in range(5000):
    xb, yb = get_batch('train')
    logits, loss = m2(xb, yb)
    optimizer2.zero_grad(set_to_none=True)
    loss.backward()
    optimizer2.step()
print("son train loss:", loss.item())

print("Görev 1 (bigram):", estimate_loss(m))
print("Görev 4 (attention):", estimate_loss(m2))

son train loss: 2.3742587566375732
Görev 1 (bigram): {'train': tensor(2.4598), 'val': tensor(2.4899)}
Görev 4 (attention): {'train': tensor(2.3863), 'val': tensor(2.4028)}
